In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/supp-figures"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import scanpy as sc

from spatial_tcr.gene_colocalization import *
from spatial_tcr.tcr import get_tcr_genes

## Load data

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/08.2-kidney_tcr_clonal_clusters_peri_glom_annot.h5ad"
adata = sc.read_h5ad(path)

# remove control samples
adata = adata[adata.obs["condition"] == "ANCA"].copy()

sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata

In [ ]:
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)

In [ ]:
ac_genes = [g for g in adata.var_names if g.startswith("TRAC")]
bc_genes = [g for g in adata.var_names if g.startswith("TRBC")]
bc_genes, ac_genes

In [ ]:
cd3_genes = [g for g in adata.var_names if g.startswith("CD3")]
cd3_genes

In [ ]:
sc.pl.dotplot(adata, var_names=cd3_genes + ac_genes + av_genes, groupby="cell_type_l1")

In [ ]:
# calculate TRAV and TRBV expression
adata.obs["TRAV_detected"] = (adata[:, av_genes].X.toarray().sum(axis=1) > 0).astype(
    int
)
adata.obs["TRBV_detected"] = (adata[:, bv_genes].X.toarray().sum(axis=1) > 0).astype(
    int
)

# correlate TRAV and TRBV with CD3X gene expression via

In [ ]:
# correlate TRAV and TRBV with CD3X gene expression via

In [ ]:
targets = sorted(set(cd3_genes + ac_genes))

# 1) combined TRAV detection
add_combined_detection(adata, av_genes, obs_key="TRAV_detected", thresh=0.0)

# 2) per-cell-type codetection with each target gene
df = codetection_by_group(
    adata,
    group_key="cell_type_l1",
    combined_key="TRAV_detected",
    target_genes=targets,
    thresh=0.0,
    min_cells=25,
)

# 3) heatmap
plot_codetect_heatmap(df, group_key="cell_type_l1", value_col="codetect_rate")

In [ ]:
# df from codetection_by_group(...)
plot_codetect_heatmap_with_stars(
    df,
    group_key="cell_type_l1",
    value_col="codetect_rate",
    p_col="q_fdr",  # or "p_fisher"
    show_ns=False,
    min_cells=25,
)

In [ ]:
df2 = add_conditional_codetect_cols(df)

# Heatmap: among TRAV+ cells, how often is CD3*/TRAC* detected?
plot_heatmap_with_optional_stars(
    df2,
    group_key="cell_type_l1",
    value_col="p_target_given_trav",
    p_col="q_fdr",
    show_stars=True,
    show_ns=False,
)

# (Optional) The inverse view:
plot_heatmap_with_optional_stars(
    df2,
    group_key="cell_type_l1",
    value_col="p_trav_given_target",
    p_col="q_fdr",
    show_stars=True,
    show_ns=False,
)

In [ ]:
# Combined TRAV detection per cell
add_combined_detection(adata, av_genes, obs_key="TRAV_detected", thresh=0.0)

# Fit per cell type, per target gene
df_lr = codetect_with_logreg_pvals(
    adata,
    celltype_key="cell_type_l1",
    trav_key="TRAV_detected",
    target_genes=targets,
    depth_key=None,  # auto-detect (or set explicitly)
    min_cells=25,
)

# Heatmap: color = codetect_rate, stars = depth-adjusted LR q-values
plot_codetect_heatmap_with_lr_stars(
    df_lr,
    celltype_key="cell_type_l1",
    value_col="codetect_rate",
    p_col="q_lr",
    show_ns=False,
)